# 03 · Washing cuboids

Drawing the liquid off wells that already hold a cuboid, so the medium can be
exchanged without lifting the cuboid out.

The whole job is one move repeated: put the tip near the bottom of a well, to
one side of the middle where the cuboid is not, and aspirate slowly. Everything
else here exists to get that position right and to keep the tip from
overflowing.

**Two different offsets, kept apart.** One is the calibration of where the well
centre really is — a fraction of a millimetre, the same for every well of the
plate. The other is a deliberate shift away from that centre, a millimetre or
two, so the tip sits by the wall instead of over the cuboid. The old code folded
them into one number (`0.25 - 2`); separated, the calibration can be re-measured
without recomputing the wash offset, and the wash offset can be tuned without
disturbing the calibration.

**Before starting:** a tip is on the pipette, the plate is in its slot, a waste
well is available, and the robot has been homed at least once since power-on.

## 0. Imports

In [ ]:
import csv
import json
from pathlib import Path

import pandas as pd

from bench_setup import *                       # noqa: F401,F403
from micropick.hardware.protocols import move_relative, move_to, require_ok, xyz

print(paths.describe())

## 1. Robot, plate and waste

The waste is wherever the drawn-off liquid goes: another well, a reservoir, a
spare plate. It only ever receives, so its calibration does not matter — the tip
empties above it.

In [ ]:
PROFILE = "lab_main"
profile = load_profile(PROFILE)

In [ ]:
openapi = ot2_api.OpentronsAPI()
openapi.add_slot_offsets([5, 8, 9], (0, 0, 64.2))

In [ ]:
r = openapi.get_run_info()

In [ ]:
openapi.toggle_lights()

In [ ]:
#Define a tip rack. This is the default tip rack for the robot.
TIP_RACK = "opentrons_96_tiprack_300ul"
#Load the tip rack. Slot = 1 by default.
r = openapi.load_labware(TIP_RACK, 11)

In [ ]:
r = openapi.pick_up_tip(openapi.labware_dct['11'], "A3")

In [ ]:
SOURCE_SLOT = 1
WASTE_SLOT  = 4
WASTE_WELL  = "B1"

SOURCE_PLATE = "corning_96_wellplate_360ul_flat"#"corning_384_wellplate_112ul_flat"    # load name
WASTE_PLATE  = "corning_6_wellplate_16.8ml_flat"

src_def = resolve_definition(SOURCE_PLATE)
print("plate:", src_def)

try:
    labware.ensure_definitions(openapi, verbose=True)
except LabwareError as exc:
    print("nothing to upload:", exc)

labware.load_labware(openapi, SOURCE_PLATE, SOURCE_SLOT)
if WASTE_SLOT != SOURCE_SLOT:
    labware.load_labware(openapi, WASTE_PLATE, WASTE_SLOT)

src_lw   = openapi.labware_dct[str(SOURCE_SLOT)]
waste_lw = openapi.labware_dct[str(WASTE_SLOT)]

for slot, lw in sorted(loaded_labware(openapi).items()):
    print(f"  slot {slot}: {lw.load_name} v{lw.version} ({lw.namespace})")

In [ ]:
openapi.drop_tip_in_place()

## 2. Measured setup

The four numbers measured on the bench live in one dictionary that is written to
disk whenever it changes and read by the moves as they happen. Nothing is
carried between cells by hand, and a kernel restart does not mean measuring
again.

In [ ]:
SETUP_PATH = paths.outputs_dir() / "wash_setup.json"

DEFAULT_SETUP = {
    "centre_offset": [0.0, 0.0],   # mm, where the well centre really is
    "wash_offset":   [-2.0, 0.0],  # mm, deliberate shift off centre
    "plate_top_z":   None,         # mm, absolute, measured by touch
    "well_depth_mm": None,         # mm, supplied by you
    "bottom_z":      None,         # mm, absolute; set to bypass top - depth
}


def load_setup(path=None) -> dict:
    path = Path(path or SETUP_PATH)
    data = dict(DEFAULT_SETUP)
    if path.exists():
        data.update(json.loads(path.read_text(encoding="utf-8")))
        print(f"loaded {path}")
    else:
        print(f"no saved setup at {path}; starting from defaults")
    return data


def save_setup(data=None, path=None) -> None:
    path = Path(path or SETUP_PATH)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data if data is not None else setup, indent=2)
                    + "\n", encoding="utf-8")


def wash_xy() -> tuple:
    """Where the tip actually goes: the calibrated centre plus the shift off it."""
    cx, cy = setup["centre_offset"]
    wx, wy = setup["wash_offset"]
    return (round(cx + wx, 3), round(cy + wy, 3))


def bottom_z() -> float:
    """Absolute Z of the well bottom, derived on every call so a re-measured
    plate top or a corrected depth cannot leave a stale number behind."""
    if setup.get("bottom_z") is not None:
        return float(setup["bottom_z"])
    top, depth = setup.get("plate_top_z"), setup.get("well_depth_mm")
    if top is None or depth is None:
        raise RuntimeError("no well bottom yet: measure the plate top and set "
                           "the depth (section 4), or set setup['bottom_z']")
    return float(top) - float(depth)


def show_setup() -> None:
    print(f"centre offset : {tuple(setup['centre_offset'])}")
    print(f"wash offset   : {tuple(setup['wash_offset'])}")
    print(f"-> tip goes to: {wash_xy()}")
    print(f"plate top     : {setup['plate_top_z']}")
    print(f"well depth    : {setup['well_depth_mm']}")
    try:
        print(f"-> well bottom: {bottom_z():.2f} mm")
    except RuntimeError as exc:
        print(f"-> well bottom: unavailable ({exc})")


setup = load_setup()
show_setup()

## 3. The well centre, and the shift off it

Park at the nominal centre, walk the tip onto the real one, store the
difference. Then choose how far to move aside for washing — far enough to clear
the cuboid, not so far that the tip rides up the wall.

In [ ]:
CHECK_WELL = "D5"          # a well with a cuboid in it, to aim by
CHECK_Z    = 0.0
LIMITS     = Limits(x=(0, 380), y=(0, 350), z=(0.1, 150))

_nominal = {}


def well_top(well, offset=(0.0, 0.0), z=CHECK_Z, direct=False):
    """Park above a well top with an xy offset. Returns the pose reached."""
    require_ok(openapi.move_to_well(src_lw, well, well_location="top",
                                    offset=(offset[0], offset[1], z),
                                    force_direct=direct),
               f"move to {well} top")
    pose = xyz(openapi)
    print(f"{well} top{tuple(round(v, 2) for v in offset)} -> "
          f"({pose[0]:.2f}, {pose[1]:.2f}, {pose[2]:.2f})")
    return pose


def nudge(axis, mm):
    pose = move_relative(openapi, axis, mm)
    print(f"({pose[0]:.2f}, {pose[1]:.2f}, {pose[2]:.2f})")
    return tuple(pose)


def jog(step=0.1, title=""):
    """Arrows for xy, q/e for z, +/- for the step, Enter to finish. The window
    must have focus; with no camera it is blank but the keys still work."""
    pos = jog_in_window(JogController(openapi, limits=LIMITS, step=step),
                        camera=None, title=title or "jog")
    print("stopped at", tuple(round(v, 2) for v in pos))
    return pos


def nominal_top(well, z=CHECK_Z):
    """Park at the well's nominal centre and remember it for capture_centre."""
    _nominal["pose"] = well_top(well, offset=(0.0, 0.0), z=z)
    return _nominal["pose"]


def capture_centre():
    """Store the current pose's departure from the last nominal centre."""
    if "pose" not in _nominal:
        raise RuntimeError("call nominal_top(well) first")
    cur, ref = xyz(openapi), _nominal["pose"]
    setup["centre_offset"] = [round(cur[0] - ref[0], 3),
                              round(cur[1] - ref[1], 3)]
    save_setup()
    print(f"centre offset -> {setup['centre_offset']}  (saved)")
    return tuple(setup["centre_offset"])


def set_centre(dx, dy):
    setup["centre_offset"] = [round(float(dx), 3), round(float(dy), 3)]
    save_setup()
    print(f"centre offset -> {setup['centre_offset']}  (saved)")


def set_wash_offset(dx, dy):
    setup["wash_offset"] = [round(float(dx), 3), round(float(dy), 3)]
    save_setup()
    print(f"wash offset -> {setup['wash_offset']}  -> tip at {wash_xy()}  (saved)")


def check_centre(well, z=CHECK_Z):
    """Park on the stored centre — what is saved, not what was typed here."""
    return well_top(well, offset=tuple(setup["centre_offset"]), z=z)

In [ ]:
# 1) park at the nominal centre
nominal_top(CHECK_WELL)

In [ ]:
# 2) walk onto the real centre: nudge("x", 0.1) / nudge("y", -0.1), or jog()
jog()

In [ ]:
# 3) store it, then look at what was stored
capture_centre()
check_centre(CHECK_WELL)

In [ ]:
# How far to move aside for washing. Negative x goes towards column 1.
# Roughly: less than the well radius, more than the cuboid's half-width.
set_wash_offset(-2.5, 0.0)

## 4. Plate top and well bottom

The top is measured by touch; the bottom is that minus the well depth. The depth
is yours to supply — not every plate has a definition with usable geometry, and
where one exists it describes the catalogue part rather than the plate on the
deck. `depth_hint` reads it when it can, only as something to check against.

Come down in small steps: the tip is long and flexes, so the touch is something
you see, not something the robot reports.

In [ ]:
def depth_hint(defn, well="A1"):
    try:
        return float(defn.data["wells"][well]["depth"])
    except (KeyError, TypeError, ValueError):
        return None


def set_well_depth(mm):
    setup["well_depth_mm"] = float(mm)
    save_setup()
    print(f"well depth -> {setup['well_depth_mm']} mm  (saved)")


def capture_plate_top():
    setup["plate_top_z"] = round(xyz(openapi)[2], 3)
    save_setup()
    print(f"plate top -> {setup['plate_top_z']} mm  (saved)")


def set_bottom_z(z=None):
    """Set the bottom directly, or None to go back to top minus depth."""
    setup["bottom_z"] = None if z is None else round(float(z), 3)
    save_setup()
    print(f"bottom override -> {setup['bottom_z']}  (saved)")


hint = depth_hint(src_def, CHECK_WELL)
print(f"the definition says {hint} mm" if hint is not None
      else "the definition gives no depth for this well")

In [ ]:
set_well_depth(13.0)         # the depth you actually want to use

In [ ]:
# Park well above the rim, on the stored centre.
check_centre(CHECK_WELL, z=5.0)

In [ ]:
# Down in steps until the tip meets the rim: nudge("z", -0.5) ... -0.1
jog()

In [ ]:
capture_plate_top()
show_setup()
move_relative(openapi, "z", 5.0)
openapi.retract_axis("leftZ")

## 5. Which wells to wash

The map is a table shaped like the plate: put a `1` in every well that holds a
cuboid. `plan_from_table` checks each filled cell against the real definition,
so a cell that is not a well of this plate is caught here rather than mid-run.

In [ ]:
dest = Destination.from_labware(SOURCE_PLATE, SOURCE_SLOT)

wash_table = empty_plate_table(dest)
wash_table.loc["B":"G", 2:4] = 1        # <- your map
# wash_table.loc["F", 4:5] = 0            # and exceptions

ORDER = "by_row"                          # "by_row" | "by_column"

planned = set(plan_from_table(wash_table, dest))
wells = [w for w in (dest.wells_by_row() if ORDER == "by_row"
                     else dest.wells_by_column()) if w in planned]
print(f"{len(wells)} wells, {wells[0]} first, {wells[-1]} last")
wash_table

## 6. The run

Per well: park above it, drop to `bottom_z() + TAKEOFF_Z` at the wash offset,
aspirate slowly, lift in small steps, then clear the well.

Slowly is the point. `FLOW_RATE` here is an order of magnitude below a transfer
rate — a fast draw pulls the cuboid to the tip whatever the offset is.

The tip holds a finite volume, so it is emptied into the waste well whenever the
next well would overflow it, and again at the end. The counter is tracked
rather than assumed, so a run stopped halfway does not leave the tip full
without saying so.

In [ ]:
# --- Parameters -----------------------------------------------------------
VOLUME      = 300.0        # ul drawn from each well
FLOW_RATE   = 10.0        # ul/s — slow on purpose
TAKEOFF_Z   = 2.5         # mm above the well bottom
APPROACH_Z  = 5.0         # mm above the well top on the way in
LIFT_STEP   = 0.1         # small lifts after aspirating
LIFT_STEPS  = 4
LIFT_CLEAR  = 20.0        # mm to rise before travelling to the next well

TIP_MAX     = 300.0       # ul the tip holds
WASTE_Z     = 5.0         # mm above the waste well top when emptying
WASTE_FLOW  = 200.0       # ul/s emptying into waste
SETTLE      = 0.2         # s

show_setup()
print(f"\ntip goes to {wash_xy()} at {bottom_z() + TAKEOFF_Z:.2f} mm, "
      f"{VOLUME} ul at {FLOW_RATE} ul/s")
print(f"{len(wells)} wells -> {VOLUME * len(wells) / 1000:.1f} ml total, "
      f"emptying every {int(TIP_MAX // VOLUME)} wells")

In [ ]:
def draw_from(well):
    """Descend to one side of the well bottom, aspirate, lift clear."""
    offset = wash_xy()
    bottom = bottom_z()
    well_top(well, offset=offset, z=APPROACH_Z)
    x, y, _ = xyz(openapi)
    move_to(openapi, (x, y, bottom + TAKEOFF_Z),
            min_z_height=bottom, force_direct=True)
    require_ok(openapi.aspirate_in_place(volume=VOLUME, flow_rate=FLOW_RATE),
               "aspirate")
    for _ in range(LIFT_STEPS):
        move_relative(openapi, "z", LIFT_STEP)
        time.sleep(SETTLE)
    move_relative(openapi, "z", LIFT_CLEAR)


def clear_tip(volume):
    """Empty the tip into the waste well and shake the hanging drop off."""
    require_ok(openapi.move_to_well(waste_lw, WASTE_WELL, well_location="top",
                                    offset=(0, 0, WASTE_Z)), "move to waste")
    if volume > 0:
        require_ok(openapi.dispense_in_place(volume=volume,
                                             flow_rate=WASTE_FLOW), "empty tip")
    if hasattr(openapi, "blow_out"):
        require_ok(openapi.blow_out(waste_lw, WASTE_WELL, well_location="top",
                                    flow_rate=WASTE_FLOW), "blow out")
    require_ok(openapi.aspirate(waste_lw, WASTE_WELL, well_location="top", volume=10, flow_rate=50), "shake up")
    require_ok(openapi.dispense(waste_lw, WASTE_WELL, well_location="top", volume=10, flow_rate=50), "shake down")

    move_relative(openapi, "z", LIFT_CLEAR)
    print(f"  tip emptied ({volume:.0f} ul)")

In [ ]:
openapi.aspirate(waste_lw, WASTE_WELL, well_location="top", volume=10, flow_rate=50)

In [ ]:
# --- One well first -------------------------------------------------------
# The risk in this procedure is lifting a cuboid, and it is invisible once the
# tip is full. Run one well, look into it, then run the rest.
clear_tip(0)
draw_from(wells[0])
print("look at the well and at the tip before going on")

In [ ]:
# If that took the cuboid with it, put it back and adjust: a larger
# wash_offset, a higher TAKEOFF_Z, or a slower FLOW_RATE.
require_ok(openapi.dispense(src_lw, wells[0], well_location="bottom",
                            offset=(*wash_xy(), TAKEOFF_Z),
                            volume=VOLUME, flow_rate=FLOW_RATE), "put back")
move_relative(openapi, "z", LIFT_CLEAR)

In [ ]:
# --- The log --------------------------------------------------------------
RUN_NAME = f"wash_{time.strftime('%Y%m%d_%H%M%S')}"
LOG_PATH = paths.logs_dir() / f"{RUN_NAME}.csv"
FIELDS = ["time", "n", "well", "volume", "in_tip", "note"]


def log_event(**row):
    fresh = not LOG_PATH.exists()
    with open(LOG_PATH, "a", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=FIELDS)
        if fresh:
            writer.writeheader()
        writer.writerow({"time": time.strftime("%Y-%m-%d %H:%M:%S"),
                         **{k: row.get(k, "") for k in FIELDS if k != "time"}})


def done_wells():
    if not LOG_PATH.exists():
        return set()
    return set(pd.read_csv(LOG_PATH)["well"].dropna())


json.dump(setup, open(paths.outputs_dir() / f"{RUN_NAME}_setup.json", "w"),
          indent=2)
print("log:", LOG_PATH)

In [ ]:
RUN_NAME = f"wash_{time.strftime('%Y%m%d_%H%M%S')}"
LOG_PATH = paths.logs_dir() / f"{RUN_NAME}.csv"

In [ ]:
# --- Run ------------------------------------------------------------------
done = done_wells()
if done:
    print(f"{len(done)} wells already washed in this log; skipping\n")

in_tip = 0.0
openapi.retract_axis("leftZ")
try:
    clear_tip(0)
    for n, well in enumerate(wells, 1):
        if well in done:
            continue
        if in_tip + VOLUME > TIP_MAX:
            clear_tip(in_tip)
            in_tip = 0.0
        draw_from(well)
        in_tip += VOLUME
        log_event(n=n, well=well, volume=VOLUME, in_tip=in_tip)
        print(f"[{n}/{len(wells)}] {well}  (tip holds {in_tip:.0f} ul)")
    if in_tip > 0:
        clear_tip(in_tip)
        in_tip = 0.0
finally:
    if in_tip > 0:
        print(f"!! {in_tip:.0f} ul left in the tip — empty it by hand")
        log_event(note=f"interrupted holding {in_tip:.0f} ul")
    openapi.retract_axis("leftZ")
    print("Z retracted")

## 7. What happened

In [ ]:
log = pd.read_csv(LOG_PATH)
washed = log["well"].dropna()
print(f"{len(washed)} wells washed, {log['volume'].sum() / 1000:.2f} ml drawn")

missed = [w for w in wells if w not in set(washed)]
print(f"not washed: {len(missed)}" + (f" — {', '.join(missed)}" if missed else ""))

seen = wash_table.copy() * 0
for w in washed:
    row, col = w[0], int(w[1:])
    if row in seen.index and col in seen.columns:
        seen.at[row, col] = 1
seen

In [ ]:
openapi.toggle_lights()